# EWAT Dataset Exploration
Quick EDA for labeled EWAT runs: signal, graph, labels.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.manifold import TSNE

sns.set_theme(style='whitegrid')

In [ ]:
DATA_ROOT = Path('../data/raw')
runs = sorted(DATA_ROOT.glob('run_*'))
if not runs:
    raise FileNotFoundError(f'No run_* directory found in {DATA_ROOT.resolve()}')
RUN_DIR = runs[-1]
RUN_DIR

In [ ]:
with np.load(RUN_DIR / 'signal.npz') as payload:
    signal = payload['signal']

with np.load(RUN_DIR / 'adjacency.npz') as payload:
    adjacency = payload['adjacency']

labels = pd.read_parquet(RUN_DIR / 'labels.parquet')
graph_stats = pd.read_csv(RUN_DIR / 'graph_stats.csv')
with open(RUN_DIR / 'services.json', encoding='utf-8') as f:
    services = json.load(f)

print('signal shape:', signal.shape)
print('adjacency shape:', adjacency.shape)
print('n services:', len(services))
labels.head()

In [ ]:
inj = labels[labels['regime'] == 'injection']
plt.figure(figsize=(11, 4))
sns.countplot(data=inj, x='scenario', order=sorted(inj['scenario'].unique()))
plt.xticks(rotation=45, ha='right')
plt.title('Scenario distribution during injection')
plt.tight_layout()

In [ ]:
flat = signal.reshape(signal.shape[0], -1)
flat = np.nan_to_num(flat, nan=0.0, posinf=0.0, neginf=0.0)
n_samples = min(len(flat), 1500)
subset = flat[:n_samples]
subset_labels = labels.iloc[:n_samples]

embedding = TSNE(n_components=2, random_state=42, init='pca', learning_rate='auto').fit_transform(subset)
proj = pd.DataFrame({
    'x': embedding[:, 0],
    'y': embedding[:, 1],
    'scenario': subset_labels['scenario'].values,
    'regime': subset_labels['regime'].values,
})

plt.figure(figsize=(8, 6))
sns.scatterplot(data=proj, x='x', y='y', hue='scenario', style='regime', s=25, alpha=0.75)
plt.title('t-SNE of signal snapshots')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()

In [ ]:
mean_volume = adjacency[:, :, :, 0].mean(axis=0) if adjacency.size else np.zeros((len(services), len(services)))
plt.figure(figsize=(10, 8))
sns.heatmap(mean_volume, cmap='magma', xticklabels=services, yticklabels=services)
plt.title('Mean edge volume heatmap')
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()